In [1]:
import pandas as pd
import numpy as np
import re
import fitz  # PyMuPDF
from pathlib import Path

In [2]:
# ---- FILE PATHS ----
PDF_PATH = ENTER PATH
RETRACTION_WATCH_CSV = ENTER PATH

In [3]:
def normalize_doi(doi):
    """
    Normalize DOI by:
    - Lowercasing
    - Removing URL prefixes
    - Removing 'doi:' prefix
    - Stripping whitespace
    """
    if pd.isna(doi):
        return np.nan
    
    doi = str(doi).strip().lower()
    
    doi = re.sub(r'https?://(dx\.)?doi\.org/', '', doi)
    doi = re.sub(r'^doi:\s*', '', doi)
    
    return doi

In [4]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

full_text = extract_text_from_pdf(PDF_PATH)

print("PDF text extracted.")

PDF text extracted.


In [5]:
def extract_references_section(text):
    """
    Attempt to isolate references section.
    """
    match = re.search(r'(references|bibliography)', text, re.IGNORECASE)
    if match:
        return text[match.start():]
    return text  # fallback if not found

references_text = extract_references_section(full_text)
lines = references_text.split("\n")

print("Reference section extracted.")

Reference section extracted.


In [6]:
def extract_dois(text_lines):
    doi_pattern = re.compile(r'10\.\d{4,9}/[-._;()/:a-z0-9]+', re.IGNORECASE)
    
    found_dois = []
    
    for line in text_lines:
        matches = doi_pattern.findall(line)
        for m in matches:
            found_dois.append(normalize_doi(m))
    
    return list(set(found_dois))  # unique

reference_dois = extract_dois(lines)

print("DOIs found in references:", len(reference_dois))
reference_dois[:10]

DOIs found in references: 235


['10.1001/jamanetworkopen.2020.14688',
 '10.1038/s41575-024-00893-5',
 '10.1016/j.chroma.2013.10.025',
 '10.15585/mmwr.mm6452a1',
 '10.1017/s1368980018003762',
 '10.1038/s41559-022-01910-z',
 '10.1007/bf02534049',
 '10.3945/an.113.004929',
 '10.1016/s0002-8223(99)00041-3',
 '10.1016/s0002-9149(00)01127-9']

In [7]:
def extract_first_authors(text_lines):
    authors = []
    
    for line in text_lines:
        if "," in line:
            first_part = line.split(",")[0]
            if len(first_part.split()) <= 3:
                authors.append(first_part.strip().lower())
    
    return list(set(authors))

reference_authors = extract_first_authors(lines)

print("First authors extracted:", len(reference_authors))
reference_authors[:10]

First authors extracted: 599


['beans',
 '201. kanazawa k',
 '202. kanazawa k',
 'food systems',
 '166. maki kc',
 'rich oils.139 importantly',
 '134. okamoto t',
 '138. chait a',
 'in 1977',
 'and aldehydes']

In [8]:
df_rw = pd.read_csv(RETRACTION_WATCH_CSV)

print("Retraction Watch dataset loaded.")
print("Columns:")
print(df_rw.columns)

Retraction Watch dataset loaded.
Columns:
Index(['Record ID', 'Title', 'Subject', 'Institution', 'Journal', 'Publisher',
       'Country', 'Author', 'URLS', 'ArticleType', 'RetractionDate',
       'RetractionDOI', 'RetractionPubMedID', 'OriginalPaperDate',
       'OriginalPaperDOI', 'OriginalPaperPubMedID', 'RetractionNature',
       'Reason', 'Paywalled', 'Notes'],
      dtype='object')


In [9]:
# Normalize DOI columns
df_rw["OriginalPaperDOI_norm"] = df_rw["OriginalPaperDOI"].apply(normalize_doi)
df_rw["RetractionDOI_norm"] = df_rw["RetractionDOI"].apply(normalize_doi)

# Normalize authors
df_rw["Author_norm"] = df_rw["Author"].astype(str).str.lower()

print("Retraction Watch DOIs normalized.")

Retraction Watch DOIs normalized.


In [10]:
rw_doi_set = set(
    df_rw["OriginalPaperDOI_norm"]
    .dropna()
    .unique()
)

matched_dois = [doi for doi in reference_dois if doi in rw_doi_set]

print("Matched DOIs:", len(matched_dois))

Matched DOIs: 0


In [11]:
rw_author_set = set(
    df_rw["Author_norm"]
    .dropna()
    .unique()
)

matched_authors = [
    author for author in reference_authors
    if any(author in rw_author for rw_author in rw_author_set)
]

print("Matched Authors:", len(matched_authors))

Matched Authors: 27


In [12]:
df_doi_matches = df_rw[
    df_rw["OriginalPaperDOI_norm"].isin(matched_dois)
].copy()

In [13]:
df_all_matches = df_doi_matches.copy()
print("Total unique retraction matches:", len(df_all_matches))

Total unique retraction matches: 0


In [14]:
if df_all_matches.empty:
    print("✅ No retracted references detected.")
else:
    print("⚠️ Retracted references detected:")
    
    display(df_all_matches[[
        "Title",
        "Author",
        "OriginalPaperDOI",
        "RetractionDOI",
        "Reason",
        "Date"
    ]])

✅ No retracted references detected.


In [15]:
print("----- SUMMARY -----")
print("Total DOIs found in references:", len(reference_dois))
print("Retracted DOI matches:", len(df_doi_matches))
print("Total retracted references detected:", len(df_all_matches))

----- SUMMARY -----
Total DOIs found in references: 235
Retracted DOI matches: 0
Total retracted references detected: 0


In [16]:
rw_dois = set(df_rw["OriginalPaperDOI_norm"].dropna())
pdf_dois = set(reference_dois)

overlap = pdf_dois.intersection(rw_dois)

print("Direct overlap count:", len(overlap))
print("Overlapping DOIs:", list(overlap)[:10])

Direct overlap count: 0
Overlapping DOIs: []
